# Day 12 — Build Your First Artificial Neural Network
## Fashion MNIST Clothing Classifier with TensorFlow / Keras

**The task:** given a 28×28 greyscale photo of a piece of clothing, decide which of 10 categories it belongs to.

**The pipeline:**

| Step | What happens |
|------|--------------|
| 1 | Load the Fashion MNIST dataset |
| 2 | Explore it — shapes, labels, sample images |
| 3 | Normalize pixel values from 0–255 to 0–1 |
| 4 | Build a simple ANN |
| 5 | Train it |
| 6 | Evaluate on unseen test data |
| 7 | Plot accuracy and loss curves |
| 8 | Predict on test images, show predicted vs actual |

Run the cells top to bottom.

---
## 0. Imports and Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hush TensorFlow's startup noise

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)

# Fixed seed so this notebook gives the same numbers every time it is run.
keras.utils.set_random_seed(42)

# Fashion MNIST labels are integers 0-9. These are what they mean.
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

---
## 1. Load the Dataset

Keras downloads Fashion MNIST once (~30 MB) and caches it in `~/.keras/datasets`.
It arrives **already split** into a training set and a test set.

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print("Training images :", X_train.shape, "  labels:", y_train.shape)
print("Test images     :", X_test.shape, "  labels:", y_test.shape)

### Reading the shape `(60000, 28, 28)`

- `60000` → number of images
- `28` → pixel height
- `28` → pixel width

There is no 4th number because these are **greyscale**. A colour dataset would be `(60000, 28, 28, 3)` — the `3` being red, green, blue.

**Why a separate test set?** Because measuring accuracy on data the model trained on tells you how well it *memorised*, not how well it *learned*.

---
## 2. Explore the Dataset

In [ ]:
print("Pixel value range :", X_train.min(), "to", X_train.max())
print("Pixel data type   :", X_train.dtype, " (uint8 = 8-bit integer, 0-255)")
print("Number of classes :", len(np.unique(y_train)))

print("\nClass distribution in the training set:")
for label in range(10):
    count = int((y_train == label).sum())
    print(f"  {label}  {CLASS_NAMES[label]:<15} {count:<8} {count/len(y_train)*100:.1f}%")

Perfectly balanced — 6,000 of each class. That means plain **accuracy** is a fair metric here. On an imbalanced dataset it would not be (a model that always predicts the majority class could score 90% while being useless).

In [ ]:
# A grid of sample training images
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"{CLASS_NAMES[y_train[i]]}\n(label {y_train[i]})", fontsize=10)
    ax.axis("off")
fig.suptitle("Fashion MNIST - Sample Training Images", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# One example of each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for label, ax in enumerate(axes.flat):
    first = np.where(y_train == label)[0][0]
    ax.imshow(X_train[first], cmap="gray")
    ax.set_title(f"{label}: {CLASS_NAMES[label]}", fontsize=11)
    ax.axis("off")
fig.suptitle("One Example of Each Class", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# What does an image actually look like to the computer?
print("Top-left 8x8 corner of the first training image:")
print(X_train[0][:8, :8])

`0` = black, `255` = white. An image is literally just a grid of numbers, and that grid is what we feed the network.

---
## 3. Normalize the Pixel Values

We divide every pixel by 255 so the range becomes 0.0–1.0.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print("New pixel range :", X_train.min(), "to", X_train.max())
print("New data type   :", X_train.dtype)

### Why divide by 255?

1. **Gradient descent behaves badly with large inputs.** Weights multiplied by values in the hundreds produce huge activations and huge gradients, so the optimizer overshoots and the loss bounces around or explodes.
2. **Keras initialises weights assuming inputs are roughly in the −1…1 range.** Feed it 0–255 and those defaults are badly calibrated.
3. **It converges faster.** Same model, same epochs — normalized input typically reaches several percentage points higher accuracy.

⚠️ **Crucially:** the test set is divided by the *same* 255. Whatever transform you apply to training data must be applied identically at inference time.

---
## 4. Build the ANN

```
Input (28×28)
    ↓
Flatten          → 784 values, 0 params
    ↓
Dense(128, relu) → 100,480 params
    ↓
Dense(64,  relu) →   8,256 params
    ↓
Dense(10, softmax) →   650 params
    ↓
10 probabilities
```

In [ ]:
model = keras.Sequential([
    # INPUT - declares that each sample is a 28x28 grid.
    layers.Input(shape=(28, 28), name="input_layer"),

    # FLATTEN - unrolls the 28x28 grid into one long vector of 784.
    # A Dense layer can only accept a flat vector, not a 2D grid.
    # 0 parameters - it only reshapes.
    layers.Flatten(name="flatten"),

    # HIDDEN LAYER 1 - 128 neurons, each looking at all 784 pixels.
    layers.Dense(128, activation="relu", name="hidden_layer_1"),

    # HIDDEN LAYER 2 - 64 neurons, compressing the representation.
    layers.Dense(64, activation="relu", name="hidden_layer_2"),

    # OUTPUT - 10 neurons, one per clothing class.
    layers.Dense(10, activation="softmax", name="output_layer"),
], name="fashion_mnist_ann")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### What each layer does

| Layer | Transform | Params | Why |
|---|---|---|---|
| **Flatten** | (28,28) → (784,) | 0 | Dense layers need a flat vector. Note this *throws away spatial structure* — that's why CNNs beat plain ANNs on images. |
| **Dense 128, ReLU** | (784,) → (128,) | 784×128 + 128 = **100,480** | Learns low-level pixel patterns: edges, blobs, textures |
| **Dense 64, ReLU** | (128,) → (64,) | 128×64 + 64 = **8,256** | Combines those into higher-level shape concepts |
| **Dense 10, Softmax** | (64,) → (10,) | 64×10 + 10 = **650** | One neuron per class; softmax makes them probabilities summing to 1 |

**Total: 109,386 parameters.** Nearly 92% of them live in the first Dense layer — typical, because the layer touching the raw high-dimensional input is always the heaviest.

### Why these compile choices?

- **Adam** — adaptive learning rate per weight. The sensible default.
- **`sparse_categorical_crossentropy`** — use this when labels are plain integers (0–9). If labels were one-hot vectors you'd use `categorical_crossentropy`. Same maths, different label format.
- **`accuracy`** — reported for us, but *not* what the optimizer minimises. It minimises the loss.

---
## 5. Train the Model

**What happens in one training step:**

1. **Forward pass** — push a batch through, get predictions
2. **Loss** — compare predictions to true labels
3. **Backward pass** (backpropagation) — compute how much each weight contributed to the error
4. **Update** — nudge every weight in the direction that reduces loss

The validation set is carved out of the **training** data, not the test data. It lets us watch for overfitting during training while keeping the test set genuinely untouched until the very end.

In [ ]:
EPOCHS = 15

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=32,
    validation_split=0.2,   # 12,000 images held back from training
    verbose=1,
)

In [ ]:
final_train_acc = history.history["accuracy"][-1]
final_val_acc   = history.history["val_accuracy"][-1]

print(f"Final training accuracy   : {final_train_acc:.4f}  ({final_train_acc*100:.2f}%)")
print(f"Final validation accuracy : {final_val_acc:.4f}  ({final_val_acc*100:.2f}%)")
print(f"Gap                       : {final_train_acc - final_val_acc:.4f}")

---
## 6. Evaluate on the Test Set

These 10,000 images were never seen during training — not even as validation data. **This is the honest number.**

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Test loss     : {test_loss:.4f}")
print(f"\nRandom guessing would score 10%. We scored {test_acc*100:.2f}%.")

---
## 7. Plot the Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs_range, history.history["accuracy"], "o-", color="#0ea5e9",
         linewidth=2, markersize=4, label="Training accuracy")
ax1.plot(epochs_range, history.history["val_accuracy"], "s--", color="#f97316",
         linewidth=2, markersize=4, label="Validation accuracy")
ax1.axhline(test_acc, color="#22c55e", linestyle=":", linewidth=2,
            label=f"Test accuracy ({test_acc:.4f})")
ax1.set_title("Model Accuracy", fontsize=14, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(loc="lower right"); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history.history["loss"], "o-", color="#0ea5e9",
         linewidth=2, markersize=4, label="Training loss")
ax2.plot(epochs_range, history.history["val_loss"], "s--", color="#f97316",
         linewidth=2, markersize=4, label="Validation loss")
ax2.set_title("Model Loss", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(loc="upper right"); ax2.grid(alpha=0.3)

fig.suptitle("Fashion MNIST ANN - Training History", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

### How to read these curves

| Pattern | Meaning |
|---|---|
| Both rising / falling together | Healthy learning |
| Train keeps improving, validation flattens | **Overfitting** — the model is memorising |
| Validation loss turning **upward** | Stop training. That's the point where the model starts getting *worse* on new data |

Look closely at the loss plot: training loss keeps dropping, but validation loss bottoms out early and then drifts up. That is textbook overfitting, and it's exactly what `EarlyStopping` or a `Dropout` layer would fix.

---
## 8. Make Predictions

`predict()` returns, for each image, **10 probabilities** — one per class.

In [ ]:
predictions = model.predict(X_test, verbose=0)
predicted_labels = predictions.argmax(axis=1)

print("Predictions array shape:", predictions.shape)
print("  -> 10,000 images, 10 probabilities each\n")

print("First test image, full probability breakdown:")
for i, prob in enumerate(predictions[0]):
    marker = "  <-- highest" if i == predicted_labels[0] else ""
    bar = "#" * int(prob * 40)
    print(f"  {i} {CLASS_NAMES[i]:<15} {prob:.6f} {bar}{marker}")

print(f"\nPredicted : {CLASS_NAMES[predicted_labels[0]]}")
print(f"Actual    : {CLASS_NAMES[y_test[0]]}")
print(f"Correct   : {'YES' if predicted_labels[0] == y_test[0] else 'NO'}")

In [ ]:
# Text table of the first 15 predictions
print(f"{'#':<4} {'Predicted':<15} {'Actual':<15} {'Confidence':<12} {'Result':<8}")
print("-" * 60)
for i in range(15):
    pred, actual = predicted_labels[i], y_test[i]
    conf = predictions[i][pred]
    ok = "CORRECT" if pred == actual else "WRONG"
    print(f"{i:<4} {CLASS_NAMES[pred]:<15} {CLASS_NAMES[actual]:<15} {conf:<12.4f} {ok:<8}")

In [ ]:
# Visual grid: green border = correct, red border = wrong
fig, axes = plt.subplots(3, 5, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    pred, actual = predicted_labels[i], y_test[i]
    conf = predictions[i][pred]
    correct = pred == actual
    ax.imshow(X_test[i], cmap="gray")
    ax.set_title(
        f"Predicted: {CLASS_NAMES[pred]}\nActual: {CLASS_NAMES[actual]}\nConfidence: {conf:.1%}",
        fontsize=9, color="#15803d" if correct else "#dc2626", fontweight="bold")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#15803d" if correct else "#dc2626")
        spine.set_linewidth(3)
fig.suptitle("Sample Predictions  (green = correct, red = wrong)",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Where does the model actually go wrong?
wrong_idx = np.where(predicted_labels != y_test)[0]
print(f"Got {len(wrong_idx):,} of {len(y_test):,} test images wrong "
      f"({len(wrong_idx)/len(y_test)*100:.2f}%)\n")

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for ax, idx in zip(axes.flat, wrong_idx[:10]):
    pred, actual = predicted_labels[idx], y_test[idx]
    ax.imshow(X_test[idx], cmap="gray")
    ax.set_title(f"Predicted: {CLASS_NAMES[pred]}\nActual: {CLASS_NAMES[actual]}",
                 fontsize=9, color="#dc2626", fontweight="bold")
    ax.axis("off")
fig.suptitle("Misclassified Examples - Where the Model Struggles",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

---
## 9. Per-Class Performance and Confusion Matrix

Overall accuracy hides a lot. Some classes are far harder than others.

In [ ]:
print(f"{'Class':<15} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
print("-" * 48)
per_class = []
for label in range(10):
    mask = y_test == label
    correct = int((predicted_labels[mask] == label).sum())
    total = int(mask.sum())
    per_class.append(correct / total)
    print(f"{CLASS_NAMES[label]:<15} {correct:<10} {total:<10} {correct/total:<10.4f}")

best  = int(np.argmax(per_class))
worst = int(np.argmin(per_class))
print(f"\nEasiest class : {CLASS_NAMES[best]} ({per_class[best]:.2%})")
print(f"Hardest class : {CLASS_NAMES[worst]} ({per_class[worst]:.2%})")

In [ ]:
# Confusion matrix, built with plain numpy (no sklearn needed)
cm = np.zeros((10, 10), dtype=int)
for actual, pred in zip(y_test, predicted_labels):
    cm[actual, pred] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted label", fontweight="bold")
ax.set_ylabel("Actual label", fontweight="bold")
ax.set_title("Confusion Matrix - Fashion MNIST ANN", fontsize=14, fontweight="bold")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Number of images")
fig.tight_layout()
plt.show()

The **diagonal** is correct predictions. Everything off the diagonal is a confusion.

The heavy off-diagonal cells cluster among **T-shirt / Pullover / Coat / Shirt** — all upper-body garments with near-identical silhouettes at 28×28 resolution. Meanwhile Trouser, Sneaker and Bag score in the high 90s because their shapes are unmistakable.

---
## 10. Save the Model

In [ ]:
model.save("outputs/fashion_mnist_ann.keras")
print("Model saved to outputs/fashion_mnist_ann.keras")

# Loading it back later:
#   loaded = keras.models.load_model("outputs/fashion_mnist_ann.keras")

---
## Summary

| Metric | Value |
|---|---|
| Architecture | Flatten → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax) |
| Total parameters | 109,386 |
| Optimizer | Adam (lr=0.001) |
| Loss | sparse_categorical_crossentropy |
| Epochs | 15 |
| **Training accuracy** | **~92.9%** |
| **Validation accuracy** | **~87.1%** |
| **Test accuracy** | **~86.3%** |

### What I learned

- An ANN treats an image as a flat list of 784 numbers, with no idea that pixel 5 sits next to pixel 6. That's a real limitation, and it's exactly what CNNs fix.
- Normalizing the input isn't optional housekeeping — it materially changes how well the model trains.
- The gap between training and test accuracy is the honest measure of overfitting. Ours is ~6.6 points, and the validation loss curve shows exactly when it started.
- Accuracy is an average that hides the interesting failures. The confusion matrix is where the real story is.

### What I'd try next

- Add `Dropout(0.2)` after each hidden layer to close the overfitting gap
- Add `EarlyStopping(patience=3, restore_best_weights=True)` to stop at the validation-loss minimum
- Swap the Dense stack for a CNN — Conv2D layers keep the spatial structure Flatten destroys, and typically push Fashion MNIST past 92%